# ICTMD Türkiye — Akademik Yayın Takip
**Kaynaklar:** ORCID API · Semantic Scholar · DergiPark

---
**Kullanım:**
1. Hücreleri sırayla çalıştırın (Çalıştır butonu veya `Shift+Enter`)
2. Tarama için **4. hücreye** gidin, yıl/ay seçin, çalıştırın
3. Sonuçlar `results_YYYY_MM.json` ve `rapor_YYYY_MM.html` olarak indirilir

In [ ]:
# ── 1. Kurulum ──────────────────────────────────────────────────────────────
!pip install requests beautifulsoup4 -q
print('✓ Bağımlılıklar kuruldu')

In [ ]:
# ── 2. Repoyu klonla / güncelle ─────────────────────────────────────────────
import os

REPO = 'https://github.com/belmak65/acadwatch.git'
DIR  = '/content/acadwatch'

if os.path.exists(DIR):
    !git -C {DIR} pull
else:
    !git clone {REPO} {DIR}

os.chdir(DIR)
print(f'✓ Çalışma dizini: {os.getcwd()}')

In [ ]:
# ── 3. Akademisyen listesini göster ─────────────────────────────────────────
import json

with open('academicians.json', encoding='utf-8') as f:
    academs = json.load(f)

print(f'Toplam {len(academs)} akademisyen\n')
for i, a in enumerate(academs, 1):
    orcid = a.get('orcid') or '—'
    kurum = a.get('kurum_veya_profil_url') or ''
    if kurum.startswith('http'): kurum = ''
    print(f'  {i:2}. {a["isim"]:<35} {kurum:<35} ORCID: {orcid}')

In [ ]:
# ── 4. TARAMA ───────────────────────────────────────────────────────────────
# Aşağıdaki parametreleri düzenleyin, ardından hücreyi çalıştırın

YIL       = 2024          # Taranacak yıl
AYLAR     = [10, 11, 12]  # Taranacak aylar (boş liste = tüm yıl)
TEST_MODU = True          # True = sadece 3 akademisyen (deneme), False = tüm liste

# ────────────────────────────────────────────────────────────────────────────
import subprocess, sys

cmd = [sys.executable, 'scan.py', '--year', str(YIL)]

if AYLAR:
    cmd += ['--months', ','.join(str(m) for m in AYLAR)]

if TEST_MODU:
    cmd.append('--test')

print('Çalıştırılan komut:', ' '.join(cmd))
print('─' * 60)

proc = subprocess.run(cmd, capture_output=False, text=True)

if proc.returncode != 0:
    print('HATA — tarama başarısız')

In [ ]:
# ── 5. JSON sonuçlarını görüntüle ───────────────────────────────────────────
import glob, json

dosyalar = sorted(glob.glob('results_*.json'))
if not dosyalar:
    print('Henüz tarama yapılmadı.')
else:
    son = dosyalar[-1]
    with open(son, encoding='utf-8') as f:
        veri = json.load(f)

    print(f'Dosya  : {son}')
    print(f'Tarama : {veri["scan_date"][:19]}')
    print(f'Filtre : {veri["filter"]}')
    print(f'Toplam : {veri["total"]} yayın')
    print()

    from collections import defaultdict
    gruplar = defaultdict(list)
    for p in veri['publications']:
        gruplar[p['academician_name']].append(p)

    TUR = {'article':'Makale','conference':'Bildiri','book':'Kitap',
           'book-chapter':'Kitap Bölümü','preprint':'Önbaskı','event':'Etkinlik','other':'Diğer'}

    for isim, yayinlar in sorted(gruplar.items(), key=lambda x: -len(x[1])):
        print(f'\n{'─'*55}')
        print(f'  {isim}  ({len(yayinlar)} yayın)')
        for p in yayinlar:
            tur   = TUR.get(p.get('type',''), p.get('type',''))
            tarih = str(p.get('year',''))
            if p.get('month'): tarih += f'/{p["month"]:02d}'
            doi   = f'  DOI:{p["doi"]}' if p.get('doi') else ''
            src   = {'orcid':'ORCID','semantic_scholar':'S2','dergipark':'DergiPark'}.get(p.get('source',''),'')
            print(f'    [{tur}] {p["title"]}')
            print(f'           {p.get("journal","")}  {tarih}  {src}{doi}')

In [ ]:
# ── 6. HTML raporu indir ────────────────────────────────────────────────────
import glob
from google.colab import files

json_dosyalar = sorted(glob.glob('results_*.json'))
html_dosyalar = sorted(glob.glob('rapor_*.html'))

if not html_dosyalar:
    print('Rapor bulunamadı. Önce tarama (4. hücre) çalıştırın.')
else:
    for d in html_dosyalar:
        print(f'İndiriliyor: {d}')
        files.download(d)
    for d in json_dosyalar:
        print(f'İndiriliyor: {d}')
        files.download(d)

In [ ]:
# ── 7. Manuel yayın ekle ────────────────────────────────────────────────────
# Akademisyen adı, yıl ve ay, tarama ile eşleşmeli

import json, uuid
from datetime import datetime
from pathlib import Path

EKLE = [
    # Buraya eklemek istediğiniz yayınları yazın:
    # {
    #     "isim":    "Cenk Güray",
    #     "baslik":  "Makale Başlığı",
    #     "tur":     "article",   # article / conference / book / book-chapter / preprint / event
    #     "yil":     2024,
    #     "ay":      11,
    #     "dergi":   "Dergi / Konferans Adı",
    #     "doi":     "10.xxxx/...",   # yoksa boş bırakın
    #     "notlar":  ""
    # },
]

if not EKLE:
    print('EKLE listesi boş — eklemek istediğiniz yayınları yukarıya yazın.')
else:
    for kayit in EKLE:
        yil, ay = kayit['yil'], kayit['ay']
        dosya = Path(f'results_{yil}_{ay:02d}.json')
        if dosya.exists():
            with open(dosya, encoding='utf-8') as f:
                veri = json.load(f)
        else:
            veri = {'year': yil, 'month': ay, 'scan_date': None,
                    'filter': {'year': yil, 'months': [ay], 'all_time': False},
                    'total': 0, 'publications': []}

        veri['publications'].append({
            'id':                   str(uuid.uuid4()),
            'academician_name':     kayit['isim'],
            'title':                kayit['baslik'],
            'type':                 kayit.get('tur', 'other'),
            'year':                 yil,
            'month':                ay,
            'journal':              kayit.get('dergi', ''),
            'doi':                  kayit.get('doi') or None,
            'url':                  kayit.get('url', ''),
            'source':               'manual',
            'is_manual':            True,
            'month_certain':        True,
            'notes':                kayit.get('notlar', ''),
            'added_at':             datetime.now().isoformat(),
        })
        veri['total'] = len(veri['publications'])

        with open(dosya, 'w', encoding='utf-8') as f:
            json.dump(veri, f, ensure_ascii=False, indent=2)
        print(f'✓ Eklendi: {kayit["isim"]} — {kayit["baslik"][:60]}')